In [ ]:
import pandas as pd
import numpy as np
import arviz as az
import json
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
from matplotlib.pyplot import cm

from emu_renewal.inputs import DATA_PATH, BASE_PATH, ANALYSIS_TYPES
from emu_renewal.outputs import melt_df_except_first_level, get_multianalysis_dispvals_from_idatas, \
    get_multianalysis_ind_spaghetti, get_multianalysis_likelihoods, get_multianalysis_procvals, get_latest_analyses
from emu_renewal.utils import melt_df_except_first_level

set_matplotlib_formats("svg")

In [ ]:
initial_countries = json.load(open(DATA_PATH / f"config/countries.json", "r"))
all_countries = initial_countries["admissions"] + initial_countries["occupancy"]

In [ ]:
countries = all_countries
n_countries = len(countries)
colours = cm.Accent.colors

In [ ]:
import pycountry
countries = [pycountry.countries.lookup(c).alpha_2 for c in all_countries]

In [ ]:
idatas = {}
proc_dfs = {}
disp_dfs = {}
likelihoods = {}
spagh_dfs = {}
for country in countries:
    country_path = BASE_PATH / "outputs/massive" / country.lower()
    analysis_times = get_latest_analyses(country_path, ANALYSIS_TYPES)
    country_idatas = {a: az.from_netcdf(country_path / a / analysis_times[a] / "idata.nc") for a in ANALYSIS_TYPES}
    idatas[country] = country_idatas
    
    # Get the melted dataframes and make the column order consistent
    proc_dfs[country] = melt_df_except_first_level(get_multianalysis_procvals(country_path, analysis_times))[ANALYSIS_TYPES]
    disp_dfs[country] = get_multianalysis_dispvals_from_idatas(country_idatas)[ANALYSIS_TYPES]
    likelihoods[country] = get_multianalysis_likelihoods(country_path, analysis_times)[ANALYSIS_TYPES]
    spagh_dfs[country] = get_multianalysis_ind_spaghetti(country_path, "process", analysis_times)[ANALYSIS_TYPES]

In [ ]:
#| fig-cap: "Comparison of posteriors by candidate mobility modifier."
like_fig, axes = plt.subplots(4, 4, figsize=[10, 10])
flat_axes = axes.ravel()
for c in range(len(countries)):
    country = countries[c]
    c_ax = sns.kdeplot(likelihoods[country], fill=True, ax=flat_axes[c], palette=colours)
    c_ax.set_title(country)
    c_ax.set_yticks([])
    if c != 0:
        flat_axes[c].get_legend().remove()
like_fig.tight_layout()
like_fig.savefig("like_fig.svg")

In [ ]:
#| fig-cap: "Comparison of dispersion values by candidate mobility modifier."
disp_fig, axes = plt.subplots(4, 4, figsize=[10, 10])
flat_axes = axes.ravel()
for c in range(len(countries)):
    country = countries[c]
    c_ax = sns.kdeplot(disp_dfs[country], fill=True, ax=flat_axes[c], palette=colours)
    c_ax.set_title(country)
    c_ax.set_yticks([])
    # c_ax.set_xlim([0.0, 0.3])
    if c != 0:
        c_ax.get_legend().remove()
disp_fig.tight_layout()
disp_fig.savefig("disp_fig.svg")

In [ ]:
#| fig-cap: "Comparison of variable process by candidate mobility modifier."
proc_fig, axes = plt.subplots(4, 4, figsize=[10, 10])
flat_axes = axes.ravel()
for c in range(len(countries)):
    country = countries[c]
    country_path = BASE_PATH / "outputs/massive" / country.lower()
    analysis_times = get_latest_analyses(country_path, ANALYSIS_TYPES)
    for a, analysis in enumerate(analysis_times):
        data = spagh_dfs[country][analysis].quantile([0.05, 0.5, 0.95], axis=1).T
        flat_axes[c].plot(data.index, data[0.5], color=colours[a], label=analysis, linewidth=3.0)
        flat_axes[c].plot(data.index, data[[0.05, 0.95]], color=colours[a], linestyle="--", linewidth=0.5)
        flat_axes[c].set_title(country)
        flat_axes[c].legend()
        flat_axes[c].set_yscale("log")
        plt.setp(flat_axes[c].xaxis.get_majorticklabels(), rotation=70)
    if c != 0:
        flat_axes[c].get_legend().remove()
proc_fig.savefig("proc_fig.svg")
proc_fig.tight_layout()

In [ ]:
#| tbl-cap: "Median likelihood values by country and candidate mobility modifier."
country_names = [pycountry.countries.lookup(c).name for c in countries]
like_medians = pd.DataFrame(columns=country_names)
for country in countries:
    like_medians[pycountry.countries.lookup(country).name] = likelihoods[country].median()
like_medians.T